# 04A — SIMCA model pre-selection on 63 bands

Goal: pre-select relevant one-class peanut SIMCA models using the full spectral range kept in the database, i.e. 63 bands.

Protocol:

- calibration: pure peanut objects from batches 1 and 2;
- validation: pure almond and pure peanut objects from batch 3;
- test: pure almond and pure peanut objects from batch 4;
- selection is based on validation only;
- test is used after selection to estimate generalization.

Two training-matrix families are kept separate:

1. **object-matrix models**: `object_mean`, `object_median`;
2. **pixel-matrix models**: `balanced_pixel_random`, `balanced_pixel_center`, `all_pixels`.

All projection and visualization are done at pixel level. Therefore, object-matrix models are fitted on object spectra but projected onto all pixels afterwards.

All result tables are saved as `.parquet`.

In [1]:
from __future__ import annotations

import sys
import json
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 300)

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [2]:
from src.io.database_h5 import load_nir_uco_h5

from src.utils import (
    save_parquet,
    save_parquet_if_nonempty,
    load_parquet,
    list_result_files,
)

from src.spectra.preprocessing_configs import normalize_preprocessing_configs
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.workflows.simca import (
    make_target_train_filters,
    run_simca_pixel_projection_grid,
    run_simca_empirical_rule_grid,
)

from src.workflows.simca_selection_utils import (
    normalize_simca_rule_columns,
    add_detection_selection_score,
    sort_detection_selection,
    add_reference_selection_scores,
    select_top_models,
    ensure_candidate_columns,
    fill_selected_config_defaults,
    summarize_parameter_tendencies,
)

from src.visualization.plot_diagnostics import (
    plot_metric_heatmap,
)

from src.visualization.plot_generic import (
    plot_bar_values,
    plot_counts_by_group,
)

%load_ext autoreload
%autoreload 2

# 1. Params & config

In [3]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

# ---------------------------------------------------------------------
# Spectral configuration
# ---------------------------------------------------------------------
# Current workflow:
#   use all non-noisy bands stored in the H5 database.
#
# Later, set USE_WAVELENGTH_WINDOW=True to test a spectral window.
USE_WAVELENGTH_WINDOW = False

WAVELENGTH_MODE = "non_noisy_all"

WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

if USE_WAVELENGTH_WINDOW:
    RESULTS_TAG = f"{int(WINDOW_MIN_NM)}_{int(WINDOW_MAX_NM)}"
else:
    RESULTS_TAG = "non_noisy_all"

# ---------------------------------------------------------------------
# Output paths
# ---------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results" / f"04A_simca_preselection_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STANDARD_GRID_SUMMARY_PATH = RESULTS_DIR / "standard_grid_summary.parquet"
STANDARD_GRID_ERRORS_PATH = RESULTS_DIR / "standard_grid_errors.parquet"

EMPIRICAL_GRID_SUMMARY_PATH = RESULTS_DIR / "empirical_cv_grid_summary.parquet"
EMPIRICAL_GRID_ERRORS_PATH = RESULTS_DIR / "empirical_cv_grid_errors.parquet"

COMBINED_GRID_SUMMARY_PATH = RESULTS_DIR / "combined_grid_summary.parquet"
SELECTED_CANDIDATE_CONFIGS_PATH = RESULTS_DIR / "selected_candidate_configs.parquet"
PRESELECTION_PROTOCOL_PATH = RESULTS_DIR / "preselection_protocol.parquet"

PCA_SELECTED_PREPROCESSINGS_PATH = (
    PROJECT_ROOT
    / "results"
    / f"03_pca_{RESULTS_TAG}"
    / "pca_selected_preprocessings.parquet"
)

# ---------------------------------------------------------------------
# Detection protocol
# ---------------------------------------------------------------------
TARGET_CLASS = "peanut"
NON_TARGET_LABEL = "non_target"
REFERENCE_CLASSES = ("almond", TARGET_CLASS)

TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=[1, 2],
)

VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": list(REFERENCE_CLASSES),
    "batch": [3],
}

# Kept only as documentation; batch 4 diagnostics should be done later if needed.
TEST_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": list(REFERENCE_CLASSES),
    "batch": [4],
}

# ---------------------------------------------------------------------
# Matrix search space
# ---------------------------------------------------------------------
RUN_ALL_PIXELS_STANDARD = False
RUN_EMPIRICAL_FOR_ALL_PIXELS = False

STANDARD_MATRIX_METHODS = [
    "object_mean",
    "object_median",
    "balanced_pixels",
]

if RUN_ALL_PIXELS_STANDARD:
    STANDARD_MATRIX_METHODS.append("all_pixels")

# ---------------------------------------------------------------------
# SIMCA rules
# ---------------------------------------------------------------------
STANDARD_RULE_NAMES = [
    "simple",
    "alternative",
    "data_driven",
    "combined_index",
]

EMPIRICAL_RULE_VARIANTS = [
    "simple_chi2",
    "data_driven_chi2",
    "alternative_chi2_fixed2",
    "simple_emp_cv",
    "data_driven_emp_cv",
    "alternative_empHQ_emp_cv",
    "alternative_empHQ_fixed2",
    "alternative_chi2_emp_cv",
    "combined_index_chi2",
]

# ---------------------------------------------------------------------
# Preprocessing search space
# ---------------------------------------------------------------------
DEFAULT_PREPROCESSING_CONFIGS = {
    "snv": ("snv",),
    "absorbance": ("absorbance",),
    "absorbance_snv": ("absorbance", "snv"),
    "absorbance_sg_smooth": ("absorbance", "sg_smooth"),
    "absorbance_sg_d1": ("absorbance", "sg_d1"),
    "absorbance_snv_sg_smooth": ("absorbance", "snv", "sg_smooth"),
    "absorbance_snv_sg_d1": ("absorbance", "snv", "sg_d1"),
    "snv_sg_d1": ("snv", "sg_d1"),
}


def _parse_preprocessing_steps(value):
    if isinstance(value, (list, tuple)):
        return tuple(str(v) for v in value)

    value = str(value)

    if "+" in value:
        return tuple(v.strip() for v in value.split("+") if v.strip())

    return (value.strip(),)


if PCA_SELECTED_PREPROCESSINGS_PATH.exists():
    pca_selected_preprocessings_df = load_parquet(PCA_SELECTED_PREPROCESSINGS_PATH)

    PREPROCESSING_CONFIGS = {
        str(row["preprocessing"]): _parse_preprocessing_steps(row["preprocessing_steps"])
        for _, row in pca_selected_preprocessings_df.drop_duplicates("preprocessing").iterrows()
    }

    print("Loaded preprocessing shortlist from PCA notebook:")
    print(PCA_SELECTED_PREPROCESSINGS_PATH)

else:
    pca_selected_preprocessings_df = pd.DataFrame()
    PREPROCESSING_CONFIGS = DEFAULT_PREPROCESSING_CONFIGS.copy()

    print("[WARNING] PCA preprocessing shortlist not found.")
    print("Using default preprocessing search space instead:")
    print(PCA_SELECTED_PREPROCESSINGS_PATH)

PREPROCESSING_CONFIGS = normalize_preprocessing_configs(PREPROCESSING_CONFIGS)

# ---------------------------------------------------------------------
# Hyperparameter search space
# ---------------------------------------------------------------------
N_COMPONENTS_VALUES = [3, 4, 5, 6, 7, 8, 10, 11, 12]
ALPHA_VALUES = [0.05, 0.01]
OBJECT_THRESHOLDS = [0.70, 0.75, 0.80, 0.85, 0.90]

M_VALUES = [40]
BALANCED_PIXEL_STRATEGY_VALUES = ["random", "center"]

SG_WINDOW_LENGTH_VALUES = [11]
SG_POLYORDER_VALUES = [2]
POSITION_DILATION_RADIUS_VALUES = [3]

DEFAULT_M = 40
DEFAULT_SG_WINDOW_LENGTH = 11
DEFAULT_SG_POLYORDER = 2

REPLACE_BALANCED_PIXELS = False
RANDOM_STATE = 42

CV_N_SPLITS = 5
CV_GROUP_COL = "object_id"

# ---------------------------------------------------------------------
# Runtime flags
# ---------------------------------------------------------------------
RUN_STANDARD_GRID = True
RUN_EMPIRICAL_CV_GRID = True

RUN_DIAGNOSTIC_PLOTS = True

N_SELECTED_PER_MATRIX_FAMILY = 15
N_SELECTED_PER_TRAINING_MATRIX = 5
N_SELECTED_OVERALL = 30

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("WAVELENGTH_MODE:", WAVELENGTH_MODE)
print("USE_WAVELENGTH_WINDOW:", USE_WAVELENGTH_WINDOW)
print("RESULTS_TAG:", RESULTS_TAG)
print("STANDARD_MATRIX_METHODS:", STANDARD_MATRIX_METHODS)
print("Number of preprocessing configs:", len(PREPROCESSING_CONFIGS))
print("PCA_SELECTED_PREPROCESSINGS_PATH:", PCA_SELECTED_PREPROCESSINGS_PATH)

Loaded preprocessing shortlist from PCA notebook:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet
DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_preselection_non_noisy_all
WAVELENGTH_MODE: non_noisy_all
USE_WAVELENGTH_WINDOW: False
RESULTS_TAG: non_noisy_all
STANDARD_MATRIX_METHODS: ['object_mean', 'object_median', 'balanced_pixels']
Number of preprocessing configs: 13
PCA_SELECTED_PREPROCESSINGS_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet


In [4]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_H5_PATH}. Run notebook 00 first.")

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

# ---------------------------------------------------------------------
# Wavelength handling
# ---------------------------------------------------------------------
if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )

    wavelength_selection_df = wavelength_selection_summary(wavelength_info)

else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

print("Number of images:", len(image_db))
print("Number of objects:", len(object_db))
print("Wavelength configuration:")
display(wavelength_config_df)

if USE_WAVELENGTH_WINDOW:
    display(wavelength_selection_df)

object_meta_df = pd.DataFrame([
    {
        "object_id": object_id,
        "source_image": obj.get("source_clean_key"),
        "source_original": obj.get("source_image"),
        "sample_kind": obj.get("sample_kind"),
        "object_nut_type": obj.get("object_nut_type"),
        "batch": obj.get("batch"),
        "split": obj.get("split"),
        "area_pixels": obj.get("area_pixels"),
        "n_pixels": obj.get("n_pixels"),
        "n_bands": obj.get("n_bands"),
    }
    for object_id, obj in object_db.items()
])

display(
    object_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
    .sort_values(["sample_kind", "object_nut_type", "batch"], na_position="last")
)

Number of images: 48
Number of objects: 1262
Wavelength configuration:


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


,sample_kind,object_nut_type,batch,n_objects
0,mixture,unknown,NaN,722
1,position_reference,peanut,1.0,47
2,position_reference,peanut,2.0,47
3,position_reference,peanut,3.0,47
4,position_reference,peanut,4.0,5
5,pure,almond,1.0,52
6,pure,almond,2.0,59
7,pure,almond,3.0,55
8,pure,almond,4.0,48
9,pure,peanut,1.0,46


In [5]:
def load_parquet_or_empty(path: Path) -> pd.DataFrame:
    """Load a parquet file if it exists, otherwise return an empty dataframe."""
    if Path(path).exists():
        return load_parquet(path)
    return pd.DataFrame()


def display_available_columns(df: pd.DataFrame, columns: list[str], n: int = 20):
    """Display only columns available in a dataframe."""
    available = [col for col in columns if col in df.columns]
    if not available:
        display(df.head(n))
    else:
        display(df[available].head(n))

# 2. Grid search

## Standard

In [6]:
if RUN_STANDARD_GRID:
    standard_summary_df, standard_results, standard_errors_df = run_simca_pixel_projection_grid(
        object_db=object_db,
        image_db=image_db,
        matrix_methods=STANDARD_MATRIX_METHODS,
        preprocessing_configs=PREPROCESSING_CONFIGS,
        rule_names=STANDARD_RULE_NAMES,
        train_filters=TRAIN_FILTERS,
        projection_filters=VALIDATION_FILTERS,
        object_thresholds=OBJECT_THRESHOLDS,
        n_components_values=N_COMPONENTS_VALUES,
        alpha_values=ALPHA_VALUES,
        m_values=M_VALUES,
        random_state=RANDOM_STATE,
        replace=REPLACE_BALANCED_PIXELS,
        wavelengths=wavelengths,
        sg_window_length_values=SG_WINDOW_LENGTH_VALUES,
        sg_polyorder_values=SG_POLYORDER_VALUES,
        position_dilation_radius_values=POSITION_DILATION_RADIUS_VALUES,
        balanced_pixel_strategy_values=BALANCED_PIXEL_STRATEGY_VALUES,
        default_m=DEFAULT_M,
        default_sg_window_length=DEFAULT_SG_WINDOW_LENGTH,
        default_sg_polyorder=DEFAULT_SG_POLYORDER,
        keep_pixel_tables=False,
        verbose=True,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )

    standard_summary_df = normalize_simca_rule_columns(
        standard_summary_df,
        model_family="standard_rule",
    )
    standard_summary_df = add_detection_selection_score(standard_summary_df)
    standard_summary_df = sort_detection_selection(standard_summary_df, add_score=False)

    save_parquet(standard_summary_df, STANDARD_GRID_SUMMARY_PATH)
    save_parquet_if_nonempty(standard_errors_df, STANDARD_GRID_ERRORS_PATH)

else:
    standard_summary_df = load_parquet_or_empty(STANDARD_GRID_SUMMARY_PATH)
    standard_errors_df = load_parquet_or_empty(STANDARD_GRID_ERRORS_PATH)

print("Standard grid summary:", standard_summary_df.shape)
print("Standard grid errors:", standard_errors_df.shape)

display_available_columns(
    standard_summary_df,
    [
        "model_family",
        "matrix_family",
        "training_matrix_id",
        "matrix_method",
        "balanced_pixel_strategy",
        "preprocessing",
        "rule",
        "rule_variant",
        "n_components",
        "alpha",
        "object_threshold",
        "balanced_accuracy",
        "target_sensitivity",
        "non_target_specificity",
        "fn_rate",
        "fp_rate",
        "selection_score",
    ],
    n=20,
)

display(standard_errors_df.head())

[1/3744] standard | matrix=object_mean | preproc=absorbance_sg_d1 | rule=simple | A=3 | alpha=0.05 | SG=(11,2) | dilation=3
[2/3744] standard | matrix=object_mean | preproc=absorbance_sg_d1 | rule=simple | A=3 | alpha=0.01 | SG=(11,2) | dilation=3
[3/3744] standard | matrix=object_mean | preproc=absorbance_sg_d1 | rule=simple | A=4 | alpha=0.05 | SG=(11,2) | dilation=3
[4/3744] standard | matrix=object_mean | preproc=absorbance_sg_d1 | rule=simple | A=4 | alpha=0.01 | SG=(11,2) | dilation=3
[5/3744] standard | matrix=object_mean | preproc=absorbance_sg_d1 | rule=simple | A=5 | alpha=0.05 | SG=(11,2) | dilation=3
[6/3744] standard | matrix=object_mean | preproc=absorbance_sg_d1 | rule=simple | A=5 | alpha=0.01 | SG=(11,2) | dilation=3
[7/3744] standard | matrix=object_mean | preproc=absorbance_sg_d1 | rule=simple | A=6 | alpha=0.05 | SG=(11,2) | dilation=3
[8/3744] standard | matrix=object_mean | preproc=absorbance_sg_d1 | rule=simple | A=6 | alpha=0.01 | SG=(11,2) | dilation=3
[9/3744]

,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,preprocessing,rule,rule_variant,n_components,alpha,object_threshold,balanced_accuracy,target_sensitivity,non_target_specificity,fn_rate,fp_rate,selection_score
0,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,simple,simple_chi2,7,0.01,0.75,0.945455,1.0,0.890909,0.0,0.109091,-0.042881
1,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,simple,simple_chi2,7,0.01,0.70,0.936364,1.0,0.872727,0.0,0.127273,-0.061666
2,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,7,0.05,0.75,0.918182,1.0,0.836364,0.0,0.163636,-0.099216
3,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,7,0.05,0.70,0.900000,1.0,0.800000,0.0,0.200000,-0.136738
4,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,data_driven,data_driven_chi2,7,0.01,0.75,0.890909,1.0,0.781818,0.0,0.218182,-0.155489
5,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,data_driven,data_driven_chi2,7,0.01,0.70,0.881818,1.0,0.763636,0.0,0.236364,-0.174233
6,standard_rule,pixel_matrix,balanced_pixel_center_m40,balanced_pixels,center,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,7,0.01,0.75,0.881818,1.0,0.763636,0.0,0.236364,-0.174233
7,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,simple,simple_chi2,6,0.01,0.75,0.872727,1.0,0.745455,0.0,0.254545,-0.192971
8,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,7,0.01,0.80,0.863636,1.0,0.727273,0.0,0.272727,-0.211703
9,standard_rule,pixel_matrix,balanced_pixel_center_m40,balanced_pixels,center,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,7,0.01,0.70,0.863636,1.0,0.727273,0.0,0.272727,-0.211703


""


## Empirical

In [7]:
if RUN_EMPIRICAL_CV_GRID:
    empirical_matrix_methods = STANDARD_MATRIX_METHODS.copy()

    if not RUN_EMPIRICAL_FOR_ALL_PIXELS:
        empirical_matrix_methods = [
            method
            for method in empirical_matrix_methods
            if method not in {"all_pixels", "pixel"}
        ]

    empirical_summary_df, empirical_results, empirical_errors_df = run_simca_empirical_rule_grid(
        object_db=object_db,
        image_db=image_db,
        train_filters=TRAIN_FILTERS,
        projection_filters=VALIDATION_FILTERS,
        preprocessing_configs=PREPROCESSING_CONFIGS,
        matrix_methods=empirical_matrix_methods,
        rule_variants=EMPIRICAL_RULE_VARIANTS,
        n_components_values=N_COMPONENTS_VALUES,
        alpha_values=ALPHA_VALUES,
        object_thresholds=OBJECT_THRESHOLDS,
        m_values=M_VALUES,
        random_state=RANDOM_STATE,
        replace=REPLACE_BALANCED_PIXELS,
        wavelengths=wavelengths,
        sg_window_length_values=SG_WINDOW_LENGTH_VALUES,
        sg_polyorder_values=SG_POLYORDER_VALUES,
        position_dilation_radius_values=POSITION_DILATION_RADIUS_VALUES,
        cv_n_splits=CV_N_SPLITS,
        group_col=CV_GROUP_COL,
        keep_pixel_tables=False,
        keep_cv_tables=False,
        verbose=True,
        balanced_pixel_strategy_values=BALANCED_PIXEL_STRATEGY_VALUES,
        default_m=DEFAULT_M,
        default_sg_window_length=DEFAULT_SG_WINDOW_LENGTH,
        default_sg_polyorder=DEFAULT_SG_POLYORDER,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )

    empirical_summary_df = normalize_simca_rule_columns(
        empirical_summary_df,
        model_family="empirical_cv_rule",
    )
    empirical_summary_df = add_detection_selection_score(empirical_summary_df)
    empirical_summary_df = sort_detection_selection(empirical_summary_df, add_score=False)

    save_parquet(empirical_summary_df, EMPIRICAL_GRID_SUMMARY_PATH)
    save_parquet_if_nonempty(empirical_errors_df, EMPIRICAL_GRID_ERRORS_PATH)

else:
    empirical_summary_df = load_parquet_or_empty(EMPIRICAL_GRID_SUMMARY_PATH)
    empirical_errors_df = load_parquet_or_empty(EMPIRICAL_GRID_ERRORS_PATH)

print("Empirical CV grid summary:", empirical_summary_df.shape)
print("Empirical CV grid errors:", empirical_errors_df.shape)

display_available_columns(
    empirical_summary_df,
    [
        "model_family",
        "matrix_family",
        "training_matrix_id",
        "matrix_method",
        "balanced_pixel_strategy",
        "preprocessing",
        "rule",
        "rule_variant",
        "n_components",
        "alpha",
        "object_threshold",
        "balanced_accuracy",
        "target_sensitivity",
        "non_target_specificity",
        "fn_rate",
        "fp_rate",
        "cv_target_rejection_rate",
        "cv_abs_rejection_error",
        "selection_score",
    ],
    n=20,
)

display(empirical_errors_df.head())


[1/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=3 | alpha=0.05 | SG=(11,2) | dilation=3

[2/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=3 | alpha=0.01 | SG=(11,2) | dilation=3

[3/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=4 | alpha=0.05 | SG=(11,2) | dilation=3

[4/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=4 | alpha=0.01 | SG=(11,2) | dilation=3

[5/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=5 | alpha=0.05 | SG=(11,2) | dilation=3

[6/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=5 | alpha=0.01 | SG=(11,2) | dilation=3

[7/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=6 | alpha=0.05 | SG=(11,2) | dilation=3

[8/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=6 | alpha=0.01 | SG=(11,2) | dilation=3

[9/936] empirical_cv | matrix=object_me

,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,preprocessing,rule,rule_variant,n_components,alpha,object_threshold,balanced_accuracy,target_sensitivity,non_target_specificity,fn_rate,fp_rate,cv_target_rejection_rate,cv_abs_rejection_error,selection_score
0,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,simple,simple_chi2,7,0.01,0.75,0.945455,1.0,0.890909,0.0,0.109091,0.062418,0.052418,-0.042881
1,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,simple,simple_chi2,7,0.01,0.70,0.936364,1.0,0.872727,0.0,0.127273,0.062418,0.052418,-0.061666
2,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,simple,simple_emp_cv,7,0.05,0.75,0.927273,1.0,0.854545,0.0,0.145455,0.050092,0.000092,-0.080445
3,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,7,0.05,0.75,0.918182,1.0,0.836364,0.0,0.163636,0.051403,0.001403,-0.099216
4,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,simple,simple_emp_cv,7,0.05,0.70,0.918182,1.0,0.836364,0.0,0.163636,0.050092,0.000092,-0.099216
5,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_d1,simple,simple_emp_cv,6,0.05,0.75,0.909091,1.0,0.818182,0.0,0.181818,0.050092,0.000092,-0.117980
6,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_d1,data_driven,data_driven_emp_cv,6,0.05,0.75,0.909091,1.0,0.818182,0.0,0.181818,0.050092,0.000092,-0.117980
7,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,data_driven,data_driven_emp_cv,7,0.05,0.75,0.909091,1.0,0.818182,0.0,0.181818,0.050092,0.000092,-0.117980
8,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,alternative,alternative_chi2_emp_cv,7,0.05,0.75,0.909091,1.0,0.818182,0.0,0.181818,0.050092,0.000092,-0.117980
9,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,7,0.05,0.70,0.900000,1.0,0.800000,0.0,0.200000,0.051403,0.001403,-0.136738


""


In [8]:
summary_parts = []

if standard_summary_df is not None and len(standard_summary_df) > 0:
    summary_parts.append(standard_summary_df)

if empirical_summary_df is not None and len(empirical_summary_df) > 0:
    summary_parts.append(empirical_summary_df)

if not summary_parts:
    raise RuntimeError("No SIMCA grid summary available. Run at least one grid search.")

combined_summary_df = pd.concat(
    summary_parts,
    ignore_index=True,
    sort=False,
)

combined_summary_df = normalize_simca_rule_columns(combined_summary_df)
combined_summary_df = add_detection_selection_score(combined_summary_df)
combined_summary_df = add_reference_selection_scores(combined_summary_df)
combined_summary_df = sort_detection_selection(combined_summary_df, add_score=False)

save_parquet(combined_summary_df, COMBINED_GRID_SUMMARY_PATH)

display_cols = [
    "model_family",
    "matrix_family",
    "training_matrix_id",
    "matrix_method",
    "balanced_pixel_strategy",
    "balanced_pixel_strategy_effective",
    "m",
    "m_effective",
    "preprocessing",
    "rule",
    "rule_variant",
    "selected_rule_name",
    "rule_for_refit",
    "limit_source",
    "n_components",
    "alpha",
    "object_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",
    "balanced_accuracy",
    "target_sensitivity",
    "non_target_specificity",
    "fn_rate",
    "fp_rate",
    "f1_score",
    "accuracy",
    "selection_score",
    "score_conservative_target",
    "score_balanced_reference",
    "score_specificity_control",
]

display_cols = [col for col in display_cols if col in combined_summary_df.columns]

print("Combined validation summary:", combined_summary_df.shape)
print("Saved:", COMBINED_GRID_SUMMARY_PATH)

display(combined_summary_df[display_cols].head(30))

Combined validation summary: (60840, 61)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_preselection_non_noisy_all\combined_grid_summary.parquet


,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,balanced_accuracy,target_sensitivity,non_target_specificity,fn_rate,fp_rate,f1_score,accuracy,selection_score,score_conservative_target,score_balanced_reference,score_specificity_control
0,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple,chi2,7,0.01,0.75,11,2,3,0.945455,1.0,0.890909,0.0,0.109091,0.946429,0.944444,-0.042881,0.727273,3.673701,2.345455
1,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,7,0.01,0.75,11,2,3,0.945455,1.0,0.890909,0.0,0.109091,0.946429,0.944444,-0.042881,0.727273,3.673701,2.345455
2,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple,chi2,7,0.01,0.70,11,2,3,0.936364,1.0,0.872727,0.0,0.127273,0.938053,0.935185,-0.061666,0.681818,3.619871,2.236364
3,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,7,0.01,0.70,11,2,3,0.936364,1.0,0.872727,0.0,0.127273,0.938053,0.935185,-0.061666,0.681818,3.619871,2.236364
4,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,7,0.05,0.75,11,2,3,0.927273,1.0,0.854545,0.0,0.145455,0.929825,0.925926,-0.080445,0.636364,3.566188,2.127273
5,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative,chi2,7,0.05,0.75,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216,0.590909,3.512648,2.018182
6,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative_chi2_fixed2,chi2,7,0.05,0.75,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216,0.590909,3.512648,2.018182
7,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,7,0.05,0.70,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216,0.590909,3.512648,2.018182
8,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_d1,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,6,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980,0.545455,3.459248,1.909091
9,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_d1,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,6,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980,0.545455,3.459248,1.909091


# 3. Models selection

In [9]:
print("Columns available in combined_summary_df:")
print(sorted(combined_summary_df.columns))

print("\nNumber of configurations by family:")
display(
    combined_summary_df
    .groupby(["model_family", "matrix_family", "training_matrix_id"], dropna=False)
    .size()
    .reset_index(name="n_configs")
    .sort_values(["model_family", "matrix_family", "training_matrix_id"])
)

print("\nTop validation configurations:")
display(combined_summary_df[display_cols].head(30))

if RUN_DIAGNOSTIC_PLOTS and len(combined_summary_df) > 0:
    plot_metric_heatmap(
        combined_summary_df,
        index_col="preprocessing",
        columns_col="selected_rule_name",
        value_col="balanced_accuracy",
        title="Validation balanced accuracy by preprocessing and SIMCA rule",
        show=True,
    )

    plot_metric_heatmap(
        combined_summary_df,
        index_col="preprocessing",
        columns_col="selected_rule_name",
        value_col="fn_rate",
        title="Validation FN rate by preprocessing and SIMCA rule",
        show=True,
    )
else:
    print("Diagnostic plots skipped.")

Columns available in combined_summary_df:
['H_emp_cv', 'Q_emp_cv', 'accuracy', 'alpha', 'alternative_chi2_emp_cv', 'alternative_empHQ_emp_cv', 'balanced_accuracy', 'balanced_pixel_strategy', 'balanced_pixel_strategy_effective', 'cv_abs_rejection_error', 'cv_expected_rejection_rate', 'cv_n_splits', 'cv_rule_limit', 'cv_target_acceptance_rate', 'cv_target_rejection_rate', 'data_driven_emp_cv', 'f1_score', 'fn', 'fn_rate', 'fp', 'fp_rate', 'limit_source', 'm', 'm_effective', 'matrix_family', 'matrix_method', 'model_family', 'n', 'n_components', 'n_cv_groups', 'n_cv_observations', 'n_projected_pixels', 'n_train_observations', 'non_target_class', 'non_target_label', 'non_target_specificity', 'object_threshold', 'position_dilation_radius', 'precision', 'preprocessing', 'preprocessing_steps', 'rule', 'rule_for_refit', 'rule_original', 'rule_token', 'rule_variant', 'rule_variant_original', 'score_balanced_reference', 'score_conservative_target', 'score_specificity_control', 'search_method', 's

,model_family,matrix_family,training_matrix_id,n_configs
0,empirical_cv_rule,object_matrix,object_mean,10530
1,empirical_cv_rule,object_matrix,object_median,10530
2,empirical_cv_rule,pixel_matrix,balanced_pixel_center_m40,10530
3,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,10530
4,standard_rule,object_matrix,object_mean,4680
5,standard_rule,object_matrix,object_median,4680
6,standard_rule,pixel_matrix,balanced_pixel_center_m40,4680
7,standard_rule,pixel_matrix,balanced_pixel_random_m40,4680



Top validation configurations:


,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,balanced_accuracy,target_sensitivity,non_target_specificity,fn_rate,fp_rate,f1_score,accuracy,selection_score,score_conservative_target,score_balanced_reference,score_specificity_control
0,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple,chi2,7,0.01,0.75,11,2,3,0.945455,1.0,0.890909,0.0,0.109091,0.946429,0.944444,-0.042881,0.727273,3.673701,2.345455
1,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,7,0.01,0.75,11,2,3,0.945455,1.0,0.890909,0.0,0.109091,0.946429,0.944444,-0.042881,0.727273,3.673701,2.345455
2,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple,chi2,7,0.01,0.70,11,2,3,0.936364,1.0,0.872727,0.0,0.127273,0.938053,0.935185,-0.061666,0.681818,3.619871,2.236364
3,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,7,0.01,0.70,11,2,3,0.936364,1.0,0.872727,0.0,0.127273,0.938053,0.935185,-0.061666,0.681818,3.619871,2.236364
4,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,7,0.05,0.75,11,2,3,0.927273,1.0,0.854545,0.0,0.145455,0.929825,0.925926,-0.080445,0.636364,3.566188,2.127273
5,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative,chi2,7,0.05,0.75,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216,0.590909,3.512648,2.018182
6,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative_chi2_fixed2,chi2,7,0.05,0.75,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216,0.590909,3.512648,2.018182
7,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,7,0.05,0.70,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216,0.590909,3.512648,2.018182
8,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_d1,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,6,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980,0.545455,3.459248,1.909091
9,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_d1,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,6,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980,0.545455,3.459248,1.909091


## Pixel matrices

### Best balanced accuracy

In [10]:
#combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') & (combined_summary_df['balanced_accuracy']>0.92)].sort_values('balanced_accuracy', ascending=False)[cols].drop_duplicates()

In [11]:
#selected_indices = combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') & (combined_summary_df['balanced_accuracy']>0.92)].sort_values('balanced_accuracy', ascending=False)[cols].drop_duplicates().index
#pixel_best_ba = combined_summary_df.filter(items=selected_indices, axis=0)

### Best fn_rate

In [12]:
#combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') & (combined_summary_df['fn']==0) & (combined_summary_df['fp_rate']<0.50)].sort_values('fp_rate')[cols].drop_duplicates().head(20)

In [13]:
#selected_indices = combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') & (combined_summary_df['fn']==0) & (combined_summary_df['fp_rate']<0.50)].sort_values('fp_rate')[cols].drop_duplicates().head(20).index
#pixel_best_fn = combined_summary_df.filter(items=selected_indices, axis=0)

### Best score

In [14]:
#combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') ].sort_values('selection_score', ascending=False)[cols].drop_duplicates().head(15)

In [15]:
#selected_indices = combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') ].sort_values('selection_score', ascending=False)[cols].drop_duplicates().head(15).index
#pixel_best_selection_score = combined_summary_df.filter(items=selected_indices, axis=0)

In [16]:
#pixel_best_ba.shape[0], pixel_best_fn.shape[0], pixel_best_selection_score.shape[0]

In [17]:
#best_pixel = pd.concat([pixel_best_ba, pixel_best_fn, pixel_best_selection_score], axis=0).drop_duplicates()
#best_pixel[cols]

In [18]:
#best_pixel.shape[0]

## Object matrices

In [19]:
#combined_summary_df[combined_summary_df['matrix_family']=='object_matrix'][cols]

### Best balanced accuracy

In [20]:
#combined_summary_df[(combined_summary_df['matrix_family']=='object_matrix') & (combined_summary_df['balanced_accuracy']>0.71)][cols].drop_duplicates().sort_values('balanced_accuracy', ascending=False)

In [21]:
# selected_indices = combined_summary_df[(combined_summary_df['matrix_family']=='object_matrix') & (combined_summary_df['balanced_accuracy']>0.71)][cols].drop_duplicates().sort_values('balanced_accuracy', ascending=False).index
# object_best_ba = combined_summary_df.filter(items=selected_indices, axis=0)
# object_best_ba.shape[0]

### Best fn_rate

In [22]:
# object_best_fn = combined_summary_df[(combined_summary_df['matrix_family']=='object_matrix') & (combined_summary_df['fn']==0)].sort_values('fp_rate')
# object_best_fn[cols]

### Best selection score

In [23]:
# object_best_score = combined_summary_df[(combined_summary_df['matrix_family']=='object_matrix') & (combined_summary_df['non_target_specificity'] > 0.2)].sort_values('selection_score', ascending=False).head(10)
# object_best_score[cols]

In [24]:
# object_best_index = pd.concat([object_best_ba, object_best_fn, object_best_score])[cols].drop_duplicates().index
# object_best = combined_summary_df.filter(items=object_best_index, axis=0).sort_values('balanced_accuracy', ascending=False)
# object_best[cols]

## Automatic selection

In [25]:
selected_candidates_df = select_top_models(
    combined_summary_df,
    n_per_training_matrix=N_SELECTED_PER_TRAINING_MATRIX,
    n_per_matrix_family=N_SELECTED_PER_MATRIX_FAMILY,
    n_overall=N_SELECTED_OVERALL,
)

selected_candidates_df = ensure_candidate_columns(selected_candidates_df)
selected_candidates_df = normalize_simca_rule_columns(selected_candidates_df)
selected_candidates_df = fill_selected_config_defaults(
    selected_candidates_df,
    default_values={
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,
        "m": DEFAULT_M,
        "sg_window_length": DEFAULT_SG_WINDOW_LENGTH,
        "sg_polyorder": DEFAULT_SG_POLYORDER,
        "position_dilation_radius": POSITION_DILATION_RADIUS_VALUES[0],
        "alpha": ALPHA_VALUES[0],
        "object_threshold": OBJECT_THRESHOLDS[0],
    },
)
selected_candidates_df = add_detection_selection_score(selected_candidates_df)
selected_candidates_df = add_reference_selection_scores(selected_candidates_df)
selected_candidates_df = sort_detection_selection(selected_candidates_df, add_score=False)

selected_candidates_df = selected_candidates_df.reset_index(drop=True)
selected_candidates_df["selected_config_id"] = [
    f"sel_{i:03d}"
    for i in range(len(selected_candidates_df))
]
selected_candidates_df["selection_split"] = "validation_batch_3"
selected_candidates_df["selection_strategy"] = "automatic_fn_fp_hierarchical"

keep_cols = [
    "selected_config_id",
    "selection_split",
    "selection_strategy",

    "model_family",
    "matrix_family",
    "training_matrix_id",
    "matrix_method",
    "balanced_pixel_strategy",
    "balanced_pixel_strategy_effective",
    "m",
    "m_effective",

    "preprocessing",
    "preprocessing_steps",

    "rule",
    "rule_variant",
    "selected_rule_name",
    "rule_for_refit",
    "limit_source",

    "target_class",
    "non_target_label",

    "n",
    "tp",
    "fn",
    "fp",
    "tn",
    "balanced_accuracy",
    "target_sensitivity",
    "non_target_specificity",
    "fn_rate",
    "fp_rate",
    "f1_score",
    "accuracy",
    "precision",

    "selection_score",
    "score_conservative_target",
    "score_balanced_reference",
    "score_specificity_control",

    "n_components",
    "alpha",
    "object_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",

    "n_train_observations",
    "n_projected_pixels",

    "cv_target_rejection_rate",
    "cv_target_acceptance_rate",
    "cv_expected_rejection_rate",
    "cv_abs_rejection_error",
    "cv_rule_limit",
]

keep_cols = [col for col in keep_cols if col in selected_candidates_df.columns]
selected_candidates_df = selected_candidates_df[keep_cols].copy()

if selected_candidates_df.empty:
    raise RuntimeError("No candidate configuration was selected.")

save_parquet(selected_candidates_df, SELECTED_CANDIDATE_CONFIGS_PATH)

print("Selected candidate configurations:", selected_candidates_df.shape)
print("Saved:", SELECTED_CANDIDATE_CONFIGS_PATH)

display(selected_candidates_df)

Selected candidate configurations: (51, 50)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_preselection_non_noisy_all\selected_candidate_configs.parquet


,selected_config_id,selection_split,selection_strategy,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,preprocessing_steps,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,target_class,non_target_label,n,tp,fn,fp,tn,balanced_accuracy,target_sensitivity,non_target_specificity,fn_rate,fp_rate,f1_score,accuracy,precision,selection_score,score_conservative_target,score_balanced_reference,score_specificity_control,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,n_train_observations,n_projected_pixels,cv_target_rejection_rate,cv_target_acceptance_rate,cv_expected_rejection_rate,cv_abs_rejection_error,cv_rule_limit
0,sel_000,validation_batch_3,automatic_fn_fp_hierarchical,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40,40,absorbance_sg_smooth,absorbance+sg_smooth,simple,simple_chi2,simple_chi2,simple,chi2,peanut,non_target,108,53,0,6,49,0.945455,1.000000,0.890909,0.000000,0.109091,0.946429,0.944444,0.898305,-0.042881,0.727273,3.673701,2.345455,7,0.01,0.75,11,2,3,3813.0,6812.0,NaN,NaN,NaN,NaN,NaN
1,sel_001,validation_batch_3,automatic_fn_fp_hierarchical,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40,40,absorbance_sg_smooth,absorbance+sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,peanut,non_target,108,53,0,6,49,0.945455,1.000000,0.890909,0.000000,0.109091,0.946429,0.944444,0.898305,-0.042881,0.727273,3.673701,2.345455,7,0.01,0.75,11,2,3,NaN,NaN,0.062418,0.937582,0.01,0.052418,1.000000
2,sel_002,validation_batch_3,automatic_fn_fp_hierarchical,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40,40,absorbance_sg_smooth,absorbance+sg_smooth,simple,simple_chi2,simple_chi2,simple,chi2,peanut,non_target,108,53,0,7,48,0.936364,1.000000,0.872727,0.000000,0.127273,0.938053,0.935185,0.883333,-0.061666,0.681818,3.619871,2.236364,7,0.01,0.70,11,2,3,3813.0,6812.0,NaN,NaN,NaN,NaN,NaN
3,sel_003,validation_batch_3,automatic_fn_fp_hierarchical,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40,40,absorbance_sg_smooth,absorbance+sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,peanut,non_target,108,53,0,7,48,0.936364,1.000000,0.872727,0.000000,0.127273,0.938053,0.935185,0.883333,-0.061666,0.681818,3.619871,2.236364,7,0.01,0.70,11,2,3,NaN,NaN,0.062418,0.937582,0.01,0.052418,1.000000
4,sel_004,validation_batch_3,automatic_fn_fp_hierarchical,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40,40,absorbance_sg_smooth,absorbance+sg_smooth,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,peanut,non_target,108,53,0,8,47,0.927273,1.000000,0.854545,0.000000,0.145455,0.929825,0.925926,0.868852,-0.080445,0.636364,3.566188,2.127273,7,0.05,0.75,11,2,3,NaN,NaN,0.050092,0.949908,0.05,0.000092,1.461738
5,sel_005,validation_batch_3,automatic_fn_fp_hierarchical,standard_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40,40,absorbance_sg_smooth,absorbance+sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative,chi2,peanut,non_target,108,53,0,9,46,0.918182,1.000000,0.836364,0.000000,0.163636,0.921739,0.916667,0.854839,-0.099216,0.590909,3.512648,2.018182,7,0.05,0.75,11,2,3,3813.0,6812.0,NaN,NaN,NaN,NaN,NaN
6,sel_006,validation_batch_3,automatic_fn_fp_hierarchical,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40,40,absorbance_sg_smooth,absorbance+sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative_chi2_fixed2,chi2,peanut,non_target,108,53,0,9,46,0.918182,1.000000,0.836364,0.000000,0.163636,0.921739,0.916667,0.854839,-0.099216,0.590909,3.512648,2.018182,7,0.05,0.75,11,2,3,NaN,NaN,0.051403,0.948597,0.05,0.001403,2.000000
7,sel_007,validation_batch_3,automatic_fn_fp_hierarchical,em

## Selected models

In [26]:
parameter_tendencies_df = summarize_parameter_tendencies(
    combined_summary_df,
    top_fraction=0.15,
    min_top_n=20,
)

print("Parameter tendencies among top-ranked validation models:")
display(parameter_tendencies_df.head(60))

Parameter tendencies among top-ranked validation models:


,parameter,value,count,rate_in_top_models,matrix_family,n_top_models
0,alpha,0.01,3363,0.737015,object_matrix,4563
1,alpha,0.05,1200,0.262985,object_matrix,4563
2,balanced_pixel_strategy,not_applicable,4061,0.889985,object_matrix,4563
3,balanced_pixel_strategy,random,502,0.110015,object_matrix,4563
4,m,NaN,4563,1.000000,object_matrix,4563
5,matrix_method,object_median,4305,0.943458,object_matrix,4563
6,matrix_method,object_mean,258,0.056542,object_matrix,4563
7,n_components,3,724,0.158668,object_matrix,4563
8,n_components,4,615,0.134780,object_matrix,4563
9,n_components,5,572,0.125356,object_matrix,4563


In [27]:
preselection_protocol_df = pd.DataFrame([{
    "db_h5_path": str(DB_H5_PATH),
    "results_dir": str(RESULTS_DIR),

    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_active_bands": int(len(wavelengths)) if wavelengths is not None else np.nan,

    "target_class": TARGET_CLASS,
    "non_target_label": NON_TARGET_LABEL,
    "reference_classes_json": json.dumps(list(REFERENCE_CLASSES)),

    "train_filters_json": json.dumps(TRAIN_FILTERS, default=str),
    "validation_filters_json": json.dumps(VALIDATION_FILTERS, default=str),
    "test_filters_json": json.dumps(TEST_FILTERS, default=str),

    "standard_matrix_methods_json": json.dumps(STANDARD_MATRIX_METHODS),
    "run_all_pixels_standard": bool(RUN_ALL_PIXELS_STANDARD),
    "run_empirical_for_all_pixels": bool(RUN_EMPIRICAL_FOR_ALL_PIXELS),

    "standard_rule_names_json": json.dumps(STANDARD_RULE_NAMES),
    "empirical_rule_variants_json": json.dumps(EMPIRICAL_RULE_VARIANTS),

    "preprocessing_configs_json": json.dumps(
        {name: list(steps) for name, steps in PREPROCESSING_CONFIGS.items()},
        default=str,
    ),
    "pca_selected_preprocessings_path": str(PCA_SELECTED_PREPROCESSINGS_PATH),
    "used_pca_preprocessing_shortlist": bool(PCA_SELECTED_PREPROCESSINGS_PATH.exists()),

    "n_components_values_json": json.dumps(N_COMPONENTS_VALUES),
    "alpha_values_json": json.dumps(ALPHA_VALUES),
    "object_thresholds_json": json.dumps(OBJECT_THRESHOLDS),
    "m_values_json": json.dumps(M_VALUES),
    "balanced_pixel_strategy_values_json": json.dumps(BALANCED_PIXEL_STRATEGY_VALUES),
    "sg_window_length_values_json": json.dumps(SG_WINDOW_LENGTH_VALUES),
    "sg_polyorder_values_json": json.dumps(SG_POLYORDER_VALUES),
    "position_dilation_radius_values_json": json.dumps(POSITION_DILATION_RADIUS_VALUES),

    "random_state": int(RANDOM_STATE),
    "replace_balanced_pixels": bool(REPLACE_BALANCED_PIXELS),
    "cv_n_splits": int(CV_N_SPLITS) if CV_N_SPLITS is not None else np.nan,
    "cv_group_col": CV_GROUP_COL,

    "n_standard_grid_rows": int(len(standard_summary_df)),
    "n_empirical_grid_rows": int(len(empirical_summary_df)),
    "n_combined_grid_rows": int(len(combined_summary_df)),
    "n_selected_candidate_configs": int(len(selected_candidates_df)),

    "standard_grid_summary_path": str(STANDARD_GRID_SUMMARY_PATH),
    "empirical_grid_summary_path": str(EMPIRICAL_GRID_SUMMARY_PATH),
    "combined_grid_summary_path": str(COMBINED_GRID_SUMMARY_PATH),
    "selected_candidate_configs_path": str(SELECTED_CANDIDATE_CONFIGS_PATH),
}])

save_parquet(preselection_protocol_df, PRESELECTION_PROTOCOL_PATH)

print("Saved preselection protocol:")
print(PRESELECTION_PROTOCOL_PATH)

display(preselection_protocol_df)

Saved preselection protocol:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_preselection_non_noisy_all\preselection_protocol.parquet


,db_h5_path,results_dir,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_active_bands,target_class,non_target_label,reference_classes_json,train_filters_json,validation_filters_json,test_filters_json,standard_matrix_methods_json,run_all_pixels_standard,run_empirical_for_all_pixels,standard_rule_names_json,empirical_rule_variants_json,preprocessing_configs_json,pca_selected_preprocessings_path,used_pca_preprocessing_shortlist,n_components_values_json,alpha_values_json,object_thresholds_json,m_values_json,balanced_pixel_strategy_values_json,sg_window_length_values_json,sg_polyorder_values_json,position_dilation_radius_values_json,random_state,replace_balanced_pixels,cv_n_splits,cv_group_col,n_standard_grid_rows,n_empirical_grid_rows,n_combined_grid_rows,n_selected_candidate_configs,standard_grid_summary_path,empirical_grid_summary_path,combined_grid_summary_path,selected_candidate_configs_path
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,non_noisy_all,False,non_noisy_all,NaN,NaN,63,peanut,non_target,"[""almond"", ""peanut""]","{""sample_kind"": [""pure""], ""object_nut_type"": [...","{""sample_kind"": [""pure""], ""object_nut_type"": [...","{""sample_kind"": [""pure""], ""object_nut_type"": [...","[""object_mean"", ""object_median"", ""balanced_pix...",False,False,"[""simple"", ""alternative"", ""data_driven"", ""comb...","[""simple_chi2"", ""data_driven_chi2"", ""alternati...","{""absorbance_sg_d1"": [""absorbance"", ""sg_d1""], ...",C:\Users\alixg\OneDrive - Université Paris-Dau...,True,"[3, 4, 5, 6, 7, 8, 10, 11, 12]","[0.05, 0.01]","[0.7, 0.75, 0.8, 0.85, 0.9]",[40],"[""random"", ""center""]",[11],[2],[3],42,False,5,object_id,18720,42120,60840,51,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...


In [28]:
print("04A_simca_preselection.ipynb completed.")
print()
print("Essential outputs:")
print(" -", STANDARD_GRID_SUMMARY_PATH)
print(" -", EMPIRICAL_GRID_SUMMARY_PATH)
print(" -", COMBINED_GRID_SUMMARY_PATH)
print(" -", SELECTED_CANDIDATE_CONFIGS_PATH)
print(" -", PRESELECTION_PROTOCOL_PATH)

if standard_errors_df is not None and len(standard_errors_df) > 0:
    print(" -", STANDARD_GRID_ERRORS_PATH)

if empirical_errors_df is not None and len(empirical_errors_df) > 0:
    print(" -", EMPIRICAL_GRID_ERRORS_PATH)

print()
print("Summary:")
print(f" - Wavelength mode: {WAVELENGTH_MODE}")
print(f" - Active bands: {len(wavelengths) if wavelengths is not None else 'unknown'}")
print(f" - Standard grid rows: {len(standard_summary_df)}")
print(f" - Empirical CV grid rows: {len(empirical_summary_df)}")
print(f" - Combined grid rows: {len(combined_summary_df)}")
print(f" - Selected candidate configs: {len(selected_candidates_df)}")
print()
print("Next notebook:")
print("04B_simca_mixture_reference_selection.ipynb")

04A_simca_preselection.ipynb completed.

Essential outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_preselection_non_noisy_all\standard_grid_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_preselection_non_noisy_all\empirical_cv_grid_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_preselection_non_noisy_all\combined_grid_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_preselection_non_noisy_all\selected_candidate_configs.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_preselection_non_noisy_all\preselection_protocol.parquet

Summary:
 - Wavelength mode: non_noisy_all
 - Active bands: 63
 - Standard grid rows: 18720
 - Empirical CV grid rows: 42120
 - Combined grid rows: 60840
 - Selected candidate configs: 51

Next notebook:
04B_simca_mixture_reference_sel